[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/3-4_RUL%EB%A1%9C_%EA%B5%90%EC%B2%B4%EC%8B%9C%EC%A0%90_%EA%B2%B0%EC%A0%95%ED%95%98%EA%B8%B0.ipynb)

# 3-4 · 언제 교체할까? — 시간 기준 vs 상태 기준 vs 예측 기준

**⏱ 90분** · 모듈 3 마무리: 열화 지표 → RUL 예측 → **교체 시점 결정**

## 🎯 이번 시간의 질문

> 예측은 **결정을 위해** 합니다. 그래서 필터를 **언제** 교체해야 할까요?

00에서 던졌던 질문으로 돌아왔습니다. 세 가지 교체 전략을 같은 필터들에 적용해 비교합니다.

| 전략 | 교체 신호 | 필요한 것 | 16시간 과정에서 |
|---|---|---|---|
| **시간 기준 (TBM, 예방정비)** | 사용 시간이 T초가 되면 | 과거 수명 기록 | 00, 1-1 과제 3 |
| **상태 기준 (CBM)** | HI가 X A 이상이 되면 | 센서 + 기준값 | 1-2, 2-2, 3-1 |
| **예측 기준 (PdM, 예지정비)** | 예측 RUL이 L초 이하가 되면 | 센서 + 고장 이력 + 모델 | 3-3 |

### 평가 규칙

- 교체 신호가 나면 교체를 준비합니다. 준비에 **30초**가 걸리므로, 신호가 **고장보다 30초 이상 먼저** 나야 교체 성공입니다. 그렇지 못하면 **고장**입니다.
- 교체에 성공해도 필터에는 아직 수명이 남아 있습니다. `고장 시각 F − 신호 시각`만큼의 수명을 **버린(낭비한)** 것입니다.
- **총비용 = 500 × 고장 수 + 버린 수명의 합(초)**. "고장 1건은 필터 수명 500초를 버리는 것만큼 손해"라고 가정합니다(생산 중단, 청소, 불량품).

⚠️ 설정(T, X, L)은 **학습용 33개 필터로** 고르고, 마지막에 **평가용 8개 필터로** 확인합니다.

**이번 시간에 익힐 패턴:** 조건을 만족하는 **첫 시각** 찾기(`[조건]` → `groupby().min()`) · 결과표 합치기 · 막대그래프로 비교

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/filter_runs.csv",
             "data/filter_windows.csv",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

## 1. 준비 (▶) — HI, 특징, RUL 예측값

3-3의 **상한 300초 랜덤 포레스트**로 모든 시점의 예측 RUL(`pred`)을 만들어 둡니다.

- 평가용 필터: 학습용 33개로 학습한 모델의 예측
- 학습용 필터: 자기 자신을 보고 학습한 모델로 예측하면 성능이 부풀려지므로, **그 필터를 뺀 나머지로 학습한 모델**로 예측합니다(5겹 교차검증, `GroupKFold`).

In [ ]:
# ▶ 그대로 실행하세요
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold

runs = pd.read_csv('data/filter_runs.csv')
windows = pd.read_csv('data/filter_windows.csv')
main = windows[windows['status'] == 'main'].copy()
main['hi'] = main.groupby('run_id')['current_a'].transform(lambda s: s.rolling(3, min_periods=1).mean())
main['rise'] = main['hi'] - main.groupby('run_id')['hi'].transform('first')
main['slope'] = main.groupby('run_id')['hi'].diff(6).fillna(0)
cols = ['hi', 'rise', 'slope', 'time_s', 'voltage_v', 'front_c']
train = main[main['split'] == 'train'].copy()
test = main[main['split'] == 'test'].copy()

model = RandomForestRegressor(min_samples_leaf=20, random_state=0)
model.fit(train[cols], train['RUL_s'].clip(upper=300))
test['pred'] = model.predict(test[cols])

train['pred'] = np.nan
for fit_idx, pred_idx in GroupKFold(n_splits=5).split(train, groups=train['run_id']):
    m = RandomForestRegressor(min_samples_leaf=20, random_state=0)
    m.fit(train.iloc[fit_idx][cols], train.iloc[fit_idx]['RUL_s'].clip(upper=300))
    train.iloc[pred_idx, train.columns.get_loc('pred')] = m.predict(train.iloc[pred_idx][cols])

train_runs = runs[runs['split'] == 'train']
test_runs = runs[runs['split'] == 'test']
print('준비 완료 · 학습용', len(train_runs), '개 / 평가용', len(test_runs), '개 필터')

### ▶ 평가 함수

필터별 **교체 신호 시각**을 넣으면 고장 수, 평균 낭비, 총비용을 계산합니다. `detail=True`로 부르면 필터별 표를 보여 줍니다.

In [ ]:
# ▶ 그대로 실행하세요
def evaluate_policy(signal, run_table, prep=30, Cf=500, detail=False):
    """signal: 필터(run_id)별 교체 신호 시각(초). 신호가 없는 필터는 비워 둔다(= 교체 못 하고 고장)."""
    t = run_table.set_index('run_id')[['failure_s']].rename(columns={'failure_s': 'F'})
    t['신호'] = signal.reindex(t.index)
    t['성공'] = (t['F'] - t['신호']) >= prep
    t['낭비(초)'] = np.where(t['성공'], t['F'] - t['신호'], 0)
    if detail:
        return t
    return pd.Series({'고장': int((~t['성공']).sum()),
                      '평균 낭비(초)': round(t.loc[t['성공'], '낭비(초)'].mean()),
                      '총비용': int(Cf * (~t['성공']).sum() + t['낭비(초)'].sum())})

## 2. 세 전략을 학습용 필터에 적용하기

### (1) 시간 기준 — "사용 T초가 되면 교체"

T는 과거 수명의 **하위 5%**(20개 중 1개만 그 전에 고장)로 정합니다. 모든 필터에 같은 교체 시각 T를 줍니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
T = train_runs['failure_s'].quantile(0.05)
tbm = pd.Series(T, index=train_runs['run_id'])
print('교체 시각 T =', round(T), '초')
evaluate_policy(tbm, train_runs)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 수명 하위 5%(141초)에 모두 교체하면 고장은 4건으로 적지만, 필터마다 평균 **417초의 수명을 버립니다.** 수명이 115~1,136초로 제각각이기 때문입니다.

### (2) 상태 기준 — "HI가 0.95 A 이상이 되면 교체"

**조건을 만족하는 첫 시각 찾기**: HI ≥ 0.95인 행만 고르고(P2) → 필터별로 가장 이른 시각(P3 `groupby().min()`).
HI가 한 번도 0.95에 닿지 않은 필터는 결과에 없으므로 "신호 없음 → 고장"으로 계산됩니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
cbm = train[train['hi'] >= 0.95].groupby('run_id')['time_s'].min()
cbm.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
evaluate_policy(cbm, train_runs)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


필터별로 어떻게 판정됐는지 봅니다.

In [ ]:
# ▶ 그대로 실행하세요
evaluate_policy(cbm, train_runs, detail=True).head(10)

💬 3번 필터는 40초 만에 HI가 0.95를 넘어(처음부터 전류가 높았음) 865초를 버렸고, 13번은 80초에 신호가 나 205초를 버렸습니다. 필터마다 판정이 다르다는 것이 시간 기준과의 차이입니다.

### (3) 예측 기준 — "예측 RUL이 150초 이하가 되면 교체"

3-3에서 모델이 **낙관적**(실제보다 길게 예측)이라는 것을 보았습니다. 준비 시간은 30초지만 여유를 두어 L = 150초로 합니다. 코드는 상태 기준과 **같은 패턴**입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
pdm = train[train['pred'] <= 150].groupby('run_id')['time_s'].min()
evaluate_policy(pdm, train_runs)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


## 3. 한눈에 비교하기

세 결과를 표 하나로 합치고, 총비용을 막대그래프로 그립니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
compare = pd.DataFrame({'시간 기준(TBM)': evaluate_policy(tbm, train_runs),
                        '상태 기준(CBM)': evaluate_policy(cbm, train_runs),
                        '예측 기준(PdM)': evaluate_policy(pdm, train_runs)})
print(compare)

plt.bar(compare.columns, compare.loc['총비용'])
plt.ylabel('총비용 (500×고장 + 낭비 초)')
plt.title('학습용 33개 필터 · 준비 시간 30초')
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 총비용: 시간 기준 14,095 · 상태 기준 8,527 · 예측 기준 8,606.

💬 **결과 해설**
- **시간 기준 → 상태/예측 기준으로 바꾸면 비용이 약 40% 줄어듭니다.** 필터마다 실제 상태를 보고 교체하므로 낭비가 417초 → 196~225초로 절반 가까이 줄었습니다. 1-3에서 하드디스크로 본 결론과 같습니다.
- **상태 기준과 예측 기준은 거의 비슷합니다.** 이 조건(준비 30초)에서는 전류 기준값 하나로도 RUL 모델만큼 좋은 결정을 할 수 있었습니다.
- 예측 기준은 낭비가 가장 적지만(196초) 고장이 7건으로 가장 많습니다. 모델의 낙관 편향 때문입니다. L을 키우면 고장이 줄고 낭비가 늘어납니다(과제 1).

그렇다면 RUL 예측은 언제 쓸모가 있을까요? → 과제 2에서 **준비 시간이 긴 경우**를 봅니다.

---
## 4. 실습 과제 (45분)

### 📝 과제 1 · 각 전략의 설정 고르기 (학습용 필터로)
각 전략의 설정을 바꿔 **총비용이 가장 작은 설정**을 찾으세요.

| 전략 | 바꿔 볼 설정 | 최적 설정 | 고장 | 총비용 |
|---|---|---|---|---|
| 시간 기준 | 분위수 0.02, 0.05, 0.1, 0.2 | | | |
| 상태 기준 | X = 0.85, 0.9, 0.95, 1.0 | | | |
| 예측 기준 | L = 100, 150, 200, 250 | | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: for x in [0.85, 0.9, 0.95, 1.0]: print(x, evaluate_policy(train[train['hi'] >= x].groupby('run_id')['time_s'].min(), train_runs).to_dict())


✏️ **나의 답:**

### 📝 과제 2 · 교체 준비에 120초가 걸린다면?
예비 필터를 창고에서 가져와야 해서 준비에 **120초**가 걸린다고 합시다. `evaluate_policy(..., prep=120)`으로 과제 1을 반복해 각 전략의 최적 설정과 총비용을 구하세요. 예측 기준은 L = **150, 200, 250, 300**으로 해 보세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 과제 1 코드의 evaluate_policy(...)에 prep=120을 추가합니다. 예측 기준의 L은 준비 시간(120초)보다 커야 의미가 있습니다.


✏️ **나의 답:** 어느 전략이 가장 싸졌나요? 준비 시간이 바뀌었을 때 각 전략의 설정을 다시 정하는 일은 얼마나 쉬운가요?

### 📝 과제 3 · 평가용 필터로 최종 확인
과제 1(준비 30초)에서 고른 설정을 **평가용 8개 필터**(`test`, `test_runs`)에 적용해 세 전략을 비교하세요. 학습용에서 본 순서가 유지되나요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 3절의 compare 코드에서 train → test, train_runs → test_runs 로 바꾸고 과제 1의 최적 설정을 넣습니다.


✏️ **나의 답:**

### 📝 과제 4 · 최종 보고서 — 정비팀장에게 보내는 교체 정책 제안 (5문장)
이 압출기 필터의 교체 정책을 제안하세요. 반드시 포함할 것:
① 추천 전략과 설정 ② 근거 숫자(비용·고장·낭비) ③ 준비 시간 조건에 따라 달라지는 점 ④ 가정과 한계 ⑤ 앞으로 모아야 할 데이터

✏️ **나의 답:**

---
## ✅ 16시간 과정 정리 — CBM의 흐름을 직접 계산해 보았습니다

```
센서·기록 데이터 → 특징(건전성 지표) → ① 이상탐지 → ② 고장진단 → ③ RUL 예측 → 정비 결정
```

| 단계 | 핵심 알고리즘 | 직접 해 본 것 | 가장 중요한 교훈 |
|---|---|---|---|
| 고장 데이터 | 고장률(AFR) | 6만 대 디스크 90일 | 분모(관찰량)를 정확히. 고장은 드물다 |
| 규칙 경보 | 값 > 기준 | SMART 197 경보, 비용 최적 기준 | 탐지 ↔ 헛경보 교환. 기준은 **비용**이 정한다 |
| 특징 | 평균·표준편차·구간 평균 | 유압 원신호 6,000점 → 특징 | **좋은 특징 > 복잡한 모델** |
| ① 이상탐지 | z-점수, T²(다변량 관리도) | 펌프 시험장치 이상 34건, 필터 열화 감지(3-1) | 고장 데이터 없이 가능. 운전 조건이 같은 충분한 정상 기록이 핵심 |
| ② 고장진단 | 결정나무, 랜덤 포레스트 | 펌프·밸브·축압기 분류 | 혼동행렬·확신도·센서 오차까지 점검 |
| ③ RUL | 추세 외삽, 회귀(상한 자르기) | 필터 막힘 잔여수명 | 먼 미래는 불확실. 결정 구간에 집중 |
| 정비 결정 | TBM vs CBM vs PdM 비용 비교 | 필터 교체 정책 | 상태를 보면 비용이 줄고, 준비 시간이 길수록 RUL이 빛난다 |

모든 단계에서 반복한 것: **① 기준 모델과 비교 ② 처음 보는 데이터로 평가 ③ 설정을 바꿔 교환 관계 확인 ④ 숫자로 근거를 대고 한계를 적기.**